# inplace-op-unsafe-warning — worked example 2: Mutating a cached parent gives wrong gradient

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `inplace-op-unsafe-warning`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """Minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries optional `.recipe`,
    `.requires_grad`, and `.grad` (the accumulated gradient at leaves)."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
        self.grad = None
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Concept

Backward rules often read a forward input value that was cached in a Recipe. For `out = x * y`, the gradient w.r.t. `x` is `grad_out * y` — it needs the original `y`. If you mutate `y` in place after the forward but before backward, the cached value changes underneath you and the gradient comes out wrong. This is exactly why in-place ops are unsafe during backprop.

## Worked solution

We demonstrate the silent corruption directly, with no guard in the way.

1. **Forward.** Compute `out = x * y` and cache the parents `(x, y)` in a Recipe on `out`, mimicking what `wrap_forward_fn` would store.
2. **Correct gradient (reference).** Right after the forward, `multiply_back0(grad_out, x, y) = grad_out * y` uses the *current, correct* `y`. We record that as the truth.
3. **Mutate y in place.** We do `y.array += 100`, the kind of thing an over-eager optimizer or buffer reuse might do.
4. **Recompute the gradient.** Now `grad_out * y.array` reads the *mutated* `y`, so it disagrees with the reference. The numbers no longer match the math the forward actually computed.

The print shows the two gradients side by side: identical math, different answers, purely because the cached value was overwritten.

In [ ]:
import numpy as np

class MiniTensor:
    def __init__(self, array):
        self.array = np.asarray(array, dtype=np.float64)

def multiply_back0(grad_out, y_arr):
    # dL/dx for out = x*y is grad_out * y — reads the cached y value
    return grad_out * y_arr

x = MiniTensor([1.0, 2.0, 3.0])
y = MiniTensor([4.0, 5.0, 6.0])
out = MiniTensor(x.array * y.array)
grad_out = np.ones_like(out.array)

# correct grad, computed while y is still intact
grad_x_correct = multiply_back0(grad_out, y.array.copy())

# now an in-place mutation overwrites the cached y
y.array += 100.0

# the reverse pass reads the corrupted cache
grad_x_corrupted = multiply_back0(grad_out, y.array)

print('correct  dL/dx:', grad_x_correct)
print('corrupted dL/dx:', grad_x_corrupted)
print('they match:', np.allclose(grad_x_correct, grad_x_corrupted))